# A third way to read the same window

[`09_dl_lstm`](09_dl_lstm.ipynb) put two readings of the 60-settlement window against each other:
NLinear, which applies one linear map to the whole window at once, and an LSTM, which walks the
window one settlement at a time and carries a state. This notebook adds a third, fitted against
the identical request contract so that the comparison is architecture and nothing else.

A **temporal convolutional network** slides a small filter along the window instead of stepping
through it. The filter here is `kernel_size: 3`, so one convolution sees three consecutive
settlements. Stacking convolutions with growing **dilations** - `1, 2, 4, 8`, meaning each
successive block skips one, three, then seven settlements between the positions it combines -
lets a shallow stack reach far back without one filter per lag. Every convolution is **causal**:
it is padded on the left and the padding is trimmed from the right, so the value at a position is
computed only from that position and earlier ones. A model that reads its own future within the
window would score well and mean nothing.

The arithmetic is worth doing once, because it is what the dilation schedule is chosen for. Each
of the four blocks applies two convolutions at its dilation, so a block extends the reach by
`2 x (3 - 1) x d`. Summed over `d` in `1, 2, 4, 8`, the receptive field is
`1 + 4 x (1 + 2 + 4 + 8) = 61` settlements against a declared lookback of 60. **The stack is
sized so the last position sees the entire window**, with one settlement to spare - and a
shorter dilation schedule would leave the earliest part of the window unreachable no matter how
long the lookback said it was.

## Where this differs from the LSTM, and why it might matter here


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 讀取同一視窗的第三種方式

[`09_dl_lstm`](09_dl_lstm.ipynb) 比較了對 60 次結算視窗的兩種讀法：NLinear 一次對整個視窗套用線性映射；LSTM 逐次結算走過視窗，並攜帶狀態。本 Notebook 加入第三種方式，依完全相同的請求契約擬合，因此比較的差異只在架構。

**時間卷積網路**不是逐步走過視窗，而是讓小型濾波器沿視窗滑動。此處濾波器是 `kernel_size: 3`，所以一次卷積看到連續三次結算。將**擴張率**遞增的卷積堆疊起來——`1, 2, 4, 8`，代表連續區塊在組合的位置之間，依序跳過一、三、七次結算——就能讓淺層堆疊向很久以前讀取，而不必每個落後期數都設一個濾波器。每次卷積都遵守**因果時間順序**：左側填補，右側修剪填補部分，因此某位置的數值只由該位置及更早資料計算。若模型在視窗內讀到自身未來，評分可能很好，卻毫無意義。

這裡值得算一次，因為擴張率時程正是為了達到這個結果。四個區塊各自在對應擴張率下套用兩次卷積，所以每個區塊將感受範圍延伸 `2 x (3 - 1) x d`。對 `d` 加總，並依 `1, 2, 4, 8` 計算，感受野為 `1 + 4 x (1 + 2 + 4 + 8) = 61` 次結算，而宣告的回溯長度為 60。**堆疊的大小讓最後一個位置能看到整個視窗**，並多出一次結算的餘裕；若擴張率時程更短，不論回溯設定多長，模型都無法讀到視窗最早的部分。

## 這與 LSTM 有何不同，以及此處為何重要
<!-- ML4T_ZH_END c0000-p000 -->

The two architectures aggregate over time in genuinely different ways, and on this data that is
not a detail.

- The LSTM's prediction is read off the state after the **last** settlement, so information from
  early in the window has to survive being carried through sixty updates to be used.
- This TCN pools its representation by **averaging over all positions** before the output layer.
  Nothing has to survive a recurrence, and a pattern that occurred early in the window
  contributes on the same footing as one that occurred late.

For a premium that mean-reverts on a timescale of days, the two are different hypotheses about
where the signal sits: at the end of the window, or spread across it. Neither is obviously right,
which is the reason to fit both rather than to pick one.

## Same contract, same gaps, same checkpoints

Everything [`09_dl_lstm`](09_dl_lstm.ipynb) establishes about the observation grid applies here
unchanged, because it is the same request contract. The grid is the 8-hour funding settlement
cadence, so a lookback of 60 is about 20 days. A window that would cross a settlement the grid
expects and the data does not have is dropped rather than imputed
(`exclude_windows_crossing_missing_expected_periods`), so `eligible_rows` in the contracts table
below, not the panel height, is the sample the model is fitted on. Training runs 100 epochs with
a checkpoint every 5, and each of the resulting 20 checkpoints is registered as its own
prediction identity.

**Learning objectives.** By the end of this notebook you will be able to:


<!-- ML4T_ZH_BEGIN c0000-p001 -->
兩種架構以真正不同的方式彙整時間資訊，而在此資料上，這不是小細節。

- LSTM 的預測來自**最後**一次結算後的狀態，因此視窗早期的資訊必須在六十次狀態更新中持續保留，才會被使用。
- 此 TCN 在輸出層前，對**所有位置取平均**以彙整表示。資訊不必經過遞迴而存留；視窗早期出現的型態，與晚期出現者有相同的貢獻地位。

若溢價在數日的時間尺度上均值回歸，兩者對訊號所在位置提出不同假設：訊號集中於視窗尾端，或分布於整段視窗。兩者都沒有明顯必然正確的理由，因此應擬合兩者，而非事先擇一。

## 相同契約、相同缺口、相同檢查點

[`09_dl_lstm`](09_dl_lstm.ipynb) 對觀測網格所確立的一切，在此都不變，因為使用相同請求契約。網格是每八小時一次的資金費用結算節奏，所以回溯 60 次結算約為 20 天。若視窗會跨過時間網格預期應有、資料卻缺少的結算時點，就會捨棄而非插補（`exclude_windows_crossing_missing_expected_periods`）。因此，下方合約表中的 `eligible_rows`，而非面板高度，才是模型實際擬合的樣本。訓練執行 100 個週期，每 5 個週期有一個檢查點；所得 20 個檢查點各自登錄為獨立預測識別值。

**學習目標。** 完成本 Notebook 後，你將能夠：
<!-- ML4T_ZH_END c0000-p001 -->

- Explain what causal padding is for, and what a convolutional sequence model would be measuring
  without it.
- Compute the receptive field of a dilated stack and check it against the declared lookback,
  rather than assuming the two agree.
- State how a convolutional model's time aggregation differs from a recurrent model's, and why
  that is a hypothesis about the data rather than an implementation choice.
- Read a resolved request and say what will be fitted, on how many eligible rows, before any
  fitting happens.

**Book reference:** Chapter 19, convolutional sequence models.

**Prerequisites:** [`03_financial_features`](03_financial_features.ipynb) and
[`04_model_based_features`](04_model_based_features.ipynb) have written the feature matrices, and
[`05_evaluation`](05_evaluation.ipynb) has established the walk-forward folds. The canonical run
uses CUDA; the reduced run in CI does not.

**What it writes:** one training run per configuration and one complete validation prediction set
per checkpoint, grouped under a named population that [`13_backtest`](13_backtest.ipynb) reads.
**Selection happens there, on validation backtest Sharpe.** Nothing here ranks anything.

<!-- ML4T_ZH_BEGIN c0000-p002 -->
- 解釋因果時間順序填補的用途，以及沒有它時，卷積序列模型究竟會衡量什麼。
- 計算擴張卷積堆疊的感受野，並與宣告的回溯長度核對，而非假定兩者一致。
- 說明卷積模型與遞迴模型在時間資訊彙整方式上的差異，以及為何這是對資料的假設，而非單純實作選擇。
- 在擬合前讀取解析後的請求，說明實際要擬合什麼，以及會使用多少合格資料列。

**書籍參考：**第 19 章，卷積序列模型。

**先備知識：**[`03_financial_features`](03_financial_features.ipynb) 與 [`04_model_based_features`](04_model_based_features.ipynb) 已寫入特徵矩陣，[`05_evaluation`](05_evaluation.ipynb) 已確立滾動向前切分。標準執行使用 CUDA；CI 中縮小規模的執行則不用。

**寫入內容：**每種設定各有一次訓練執行，每個檢查點各有一套完整驗證預測結果，歸於具名模型群組，[`13_backtest`](13_backtest.ipynb) 會讀取。**選擇在該處依驗證回測夏普比率進行。**此處不對任何結果排名。
<!-- ML4T_ZH_END c0000-p002 -->


In [1]:
import os

import polars as pl

from case_studies.crypto_perps_funding.research_workflow import (
    REGRESSION_LABELS,
    declared_contracts,
    freeze_official_model_population,
    model_request_catalog,
    open_study,
    plan_model_catalog,
    plan_specs,
    run_model_plan,
)
from case_studies.research import population_supersedes

In [2]:
EXECUTION_TIER = "canonical"
SUPERSEDES_POPULATION: str = "1b444ce334d4"
# The generation of this notebook's own checkpoint population that this run replaces, if any.
# Distinct from SUPERSEDES_POPULATION above, which is the case-wide official model population:
# the two are separate declarations and a refit can move either without moving the other.
SUPERSEDES_MODEL_POPULATION: str = "ee303a0e10e2"
WORKSPACE = os.environ.get("ML4T_OUTPUT_DIR", "")
LABELS = REGRESSION_LABELS
PREVIEW_REDUCTIONS = {}
OVERRIDES = {"device": "cuda"}

## 1. Resolve the sequence and checkpoint identities

Nothing is fitted below. The catalog is filtered to `config_prefix="tcn"`, which is what confines
this notebook to the convolutional configurations declared in
`config/training/fwd_ret_8h.yaml` alongside the two that
[`09_dl_lstm`](09_dl_lstm.ipynb) fits.

The contracts table reads `gap_policy` and `lookback` back out of the frozen specification rather
than restating the configuration file, so it cannot describe something other than what the fit
will use. Check the lookback against the receptive field computed in the header before running
anything: if a future edit shortens the dilation schedule, the two stop agreeing and the window
grows a region the model cannot see.

<!-- ML4T_ZH_BEGIN c0003-p000 -->
## 1. 解析序列與檢查點識別值

下方不擬合任何模型。目錄篩選為 `config_prefix="tcn"`，使本 Notebook 只處理 `config/training/fwd_ret_8h.yaml` 中宣告的卷積模型設定；同處宣告的另外兩種架構由 [`09_dl_lstm`](09_dl_lstm.ipynb) 擬合。

合約表直接從已固定規格讀回 `gap_policy` 與 `lookback`，而非重新敘述設定檔，因此不會描述與實際擬合內容不同的設定。執行前，應將回溯長度與開頭算出的感受野相比：若日後編輯縮短擴張率時程，兩者就不再一致，視窗會出現模型無法讀取的區域。
<!-- ML4T_ZH_END c0003-p000 -->


In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
official_population = (
    freeze_official_model_population(
        study,
        supersedes=population_supersedes(
            study,
            name="crypto-validation-predictions-v1",
            declared=SUPERSEDES_POPULATION,
        ),
    )
    if EXECUTION_TIER == "canonical"
    else None
)
requests = model_request_catalog("deep_learning", labels=LABELS, config_prefix="tcn")
requests

family,label,config_name
str,str,str
"""deep_learning""","""fwd_ret_8h""","""tcn"""


In [4]:
plan = plan_model_catalog(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides=OVERRIDES,
    preview_reductions=PREVIEW_REDUCTIONS,
)
# Sequence eligibility follows from the resolved gap policy and lookback, so read both from the
# frozen specification instead of restating the configuration file here.
resolved_preprocessing = [spec["computation"]["preprocessing"] for spec in plan_specs(plan)]
contracts = declared_contracts(plan).with_columns(
    pl.Series("gap_policy", [step["gap_policy"] for step in resolved_preprocessing]),
    pl.Series("lookback", [step["lookback"] for step in resolved_preprocessing]),
)
contracts.select(
    "label",
    "config_name",
    "gap_policy",
    "lookback",
    "checkpoint_value",
    "eligible_rows",
    "training_hash",
)

label,config_name,gap_policy,lookback,checkpoint_value,eligible_rows,training_hash
str,str,str,i64,i64,i64,str
"""fwd_ret_8h""","""tcn""","""calendar_grid_observation_mask…",60,5,32320,"""b6fa5e808e46"""
"""fwd_ret_8h""","""tcn""","""calendar_grid_observation_mask…",60,10,32320,"""b6fa5e808e46"""
"""fwd_ret_8h""","""tcn""","""calendar_grid_observation_mask…",60,15,32320,"""b6fa5e808e46"""
"""fwd_ret_8h""","""tcn""","""calendar_grid_observation_mask…",60,20,32320,"""b6fa5e808e46"""
"""fwd_ret_8h""","""tcn""","""calendar_grid_observation_mask…",60,25,32320,"""b6fa5e808e46"""
…,…,…,…,…,…,…
"""fwd_ret_8h""","""tcn""","""calendar_grid_observation_mask…",60,80,32320,"""b6fa5e808e46"""
"""fwd_ret_8h""","""tcn""","""calendar_grid_observation_mask…",60,85,32320,"""b6fa5e808e46"""
"""fwd_ret_8h""","""tcn""","""calendar_grid_observation_mask…",60,90,32320,"""b6fa5e808e46"""


The complete case-wide population is recorded before the first fit, so a member that later
fails to train cannot quietly disappear from the population it was declared in. This notebook
produces one slice of it, and that slice must lie inside the declaration.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
完整的個案研究模型群組在首次擬合前就已記錄，因此後續訓練失敗的成員，不能默默從已宣告群組中消失。本 Notebook 產生其中一部分，而且必須包含在完整宣告中。
<!-- ML4T_ZH_END c0006-p000 -->


In [5]:
if official_population is not None:
    outside = set(plan.expected_prediction_hashes) - set(official_population.members)
    if outside:
        raise RuntimeError(
            f"{len(outside)} declared checkpoints lie outside the official model population"
        )

## 2. Execute the declared population

Each configuration is fitted on each fold, a checkpoint is persisted every fifth epoch, and one
complete validation prediction set is registered per checkpoint. The completeness check is not a
formality: a prediction set covering most of its fold's eligible keys is a different sample, not
a slightly worse result, and comparing it against a complete one in the backtest would be
comparing two models measured on different data. The run raises rather than publishing one.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 2. 執行已宣告的模型群組

每種設定都在每個切分上擬合，每五個訓練週期持久化一次檢查點，並為每個檢查點登錄一套完整驗證預測結果。完整性檢查並非形式上的步驟：若某預測集只涵蓋所屬切分的大部分合格鍵值，它使用的就是不同樣本，而非結果稍差；若在回測中將它與完整預測集比較，就等於比較在不同資料上衡量的兩個模型。執行過程會報錯，而非發布這種預測集。
<!-- ML4T_ZH_END c0008-p000 -->


In [6]:
execution = run_model_plan(
    plan,
    supersedes=population_supersedes(
        study,
        name="crypto-tcn-validation-predictions-v1",
        declared=SUPERSEDES_MODEL_POPULATION,
    ),
    population_name="crypto-tcn-validation-predictions-v1"
    if EXECUTION_TIER == "canonical"
    else None,
)
catalog = execution.catalog_rows.sort("label", "config_name", "checkpoint_value")
if (
    catalog.height != len(plan.expected_prediction_hashes)
    or catalog.filter(~pl.col("complete")).height
):
    raise RuntimeError("TCN checkpoint population is incomplete")
catalog.select(
    "label",
    "config_name",
    "checkpoint_value",
    "training_hash",
    "prediction_hash",
    "complete",
)

Fold-major CV: 2 folds × 1 configs × 60 lookback

  Fold 0: creating sequences...
    train=21,457 seq across 16 symbols
    val=15,323 seq across 18 symbols
    creating datasets...
    datasets ready
    tcn:


      epoch   1/100: train_loss=0.024097


      epoch   2/100: train_loss=0.006498


      epoch   3/100: train_loss=0.004890


      epoch   4/100: train_loss=0.004168


      epoch   5/100: train_loss=0.003970, val_loss=0.003239, IC=-0.0053


... (218 outputs left out)

direction AUC against fwd_dir_8h not computed: cannot align join key 'timestamp': predictions are Datetime(time_unit='us', time_zone='UTC'), direction label 'fwd_dir_8h' is Datetime(time_unit='ms', time_zone='UTC')


direction AUC against fwd_dir_8h not computed: cannot align join key 'timestamp': predictions are Datetime(time_unit='us', time_zone='UTC'), direction label 'fwd_dir_8h' is Datetime(time_unit='ms', time_zone='UTC')


direction AUC against fwd_dir_8h not computed: cannot align join key 'timestamp': predictions are Datetime(time_unit='us', time_zone='UTC'), direction label 'fwd_dir_8h' is Datetime(time_unit='ms', time_zone='UTC')


label,config_name,checkpoint_value,training_hash,prediction_hash,complete
str,str,i64,str,str,bool
"""fwd_ret_8h""","""tcn""",5,"""b6fa5e808e46""","""9343efca9249""",true
"""fwd_ret_8h""","""tcn""",10,"""b6fa5e808e46""","""c6bc9f0dccae""",true
"""fwd_ret_8h""","""tcn""",15,"""b6fa5e808e46""","""d639ec329cca""",true
"""fwd_ret_8h""","""tcn""",20,"""b6fa5e808e46""","""6ee68386afc4""",true
"""fwd_ret_8h""","""tcn""",25,"""b6fa5e808e46""","""bbc0508dfd2a""",true
…,…,…,…,…,…
"""fwd_ret_8h""","""tcn""",80,"""b6fa5e808e46""","""6727fa9a56fe""",true
"""fwd_ret_8h""","""tcn""",85,"""b6fa5e808e46""","""252d213aad3f""",true
"""fwd_ret_8h""","""tcn""",90,"""b6fa5e808e46""","""beaf091fbab7""",true


## Key takeaways and limitations

- **The receptive field is a property of the architecture, not of the lookback.** Four blocks at
  dilations 1, 2, 4, 8 with kernel 3 reach 61 settlements; the lookback is 60. Change either
  without checking the other and the model quietly stops seeing part of the window it is handed.
- **Causal padding is what makes the number honest.** Without trimming the right-hand padding,
  each position would be computed partly from later ones, and the validation score would be
  measuring a model that had seen the answer.
- **Averaging over positions is a hypothesis.** This TCN pools its representation across the whole
  window, so it treats a pattern early in the window as no less usable than one at the end. The
  LSTM in [`09_dl_lstm`](09_dl_lstm.ipynb) does the opposite. Which is right is an empirical
  question about where in the window the premium's information sits, and the backtest is where it
  gets answered.
- **Batch normalization pools across windows, not across time within one.** The statistics used to
  normalize a training window come from the other windows in its batch, which may be
  chronologically later within the same fold. Fold boundaries are respected, so no validation
  information reaches training - but the training objective is not a pure per-window causal
  function, and that is worth knowing before attributing a result entirely to the convolutions.
- **Two folds is what the history supports.** The reach of the dilation schedule is not the
  binding constraint on what this model can learn here; the length of the usable perpetual
  funding record is.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 重點摘要與限制

- **感受野是架構的性質，而非回溯長度的性質。** 四個區塊的擴張率為 1、2、4、8，卷積核大小為 3，可涵蓋 61 次結算；回溯長度為 60。若修改其中一項卻不檢查另一項，模型可能默默失去讀取部分輸入視窗的能力。
- **因果時間順序填補使評分可信。** 若不修剪右側填補部分，每個位置都可能部分根據較晚位置計算，驗證評分就會衡量已看過答案的模型。
- **對所有位置取平均是一項假設。** 此 TCN 在整段視窗上彙整表示，因此視窗早期出現的型態與尾端出現者同樣可用。[`09_dl_lstm`](09_dl_lstm.ipynb) 的 LSTM 採相反方式。何者正確，取決於溢價資訊位於視窗何處，須由資料回答；回測將檢驗。
- **批次正規化跨視窗彙整，而非在單一視窗內跨時間彙整。** 正規化某訓練視窗所用的統計量，來自同一批次的其他視窗，其中有些在相同切分內的時間可能較晚。切分邊界仍獲遵守，因此驗證資訊不會進入訓練；但訓練目標並非純粹逐視窗遵守因果時間順序的函數，因此不能在解讀結果時將全部效果歸因於卷積。
- **歷史資料只支撐兩個切分。** 此處限制模型學習能力的主要因素，不是擴張率時程的涵蓋範圍，而是可用的永續合約資金費率紀錄長度。
<!-- ML4T_ZH_END c0010-p000 -->
